**Question:**
What is the special property of Monic Chebyshev Polynomial that is useful to reduction of error in interpolating polynomial. Explain in your own words with justified statements that how monic Chebyshev Polynomials hold this special property.


Consider $R_n=\frac{f^{(n+1)}(ɤ)}{(n+1)!}(x-x_{0})(x-x_{1})....(x-x_{n})$

for $P_n ≈ f(x)$
How monic Polynomial reduces this error. Explain in your own words.



**Answer:**
Monic Chebyshev Polynomials have a very special property that they are the minimum monic polynomial possible for that degree for suppose there is a monic polynomial of degree n then it will be equal to Chebyshev Polynomial if it is the minimum monic polynomial of that degree.
We know it by the fact that:

$ max_{x \in [-1,1]} |T'_n(x)|=\frac{1}{2^{n-1}} $

and even the minimum polynomials are greater than this $\frac{1}{2^{n-1}}$

therefore chebyshev polynomials hold this property.


when we have

$R_n=\frac{f^{(n+1)}(ɤ)}{(n+1)!}(x-x_{0})(x-x_{1})....(x-x_{n})$

here if we divide this expression into parts in the first part is the  fraction $\frac{f^{(n+1)}(ɤ)}{(n+1)!}$ and in the second part is the monic polynomial $(x-x_{0})(x-x_{1})....(x-x_{n})$
here we cannot change the first fraction part but to get the minimum error what we do we take the second part equal to the chebyshev polynomials as it is the minimum monic polynomial it will give the minimum value and it will decrease it to the minimum value.

apart from that we can use the Chebyshev Property of $x_{i's}$ to decrease the error even further and get even smaller value.




In [ ]:
import sympy as sp
import numpy as np

def find_pivot(b, i):
    N = b.shape[0]  # Number of rows
    # Initialize the index and max value for the pivot element
    pivot_index = i
    max_value = abs(b[i, i])

    # Loop through each row from i to N-1 to find the maximum absolute value
    for k in range(i + 1, N):
        if abs(b[k, i]) > max_value:
            max_value = abs(b[k, i])
            pivot_index = k

    return pivot_index

# Define the symbolic variable
x = sp.symbols('x')

# Define the function f for symbolic exponential calculations
def f(k):
    return sp.exp(k)

# Define the PA function for handling symbolic derivations and forming a system of equations
def PA(f, n, m, DF):
    N = m + n
    a = [0] * N
    p = [0] * n
    q = [0] * m
    b = np.zeros((N, N))  # Updated size of matrix b

    # Calculate coefficients using symbolic derivatives evaluated at x = 0
    for i in range(N):
        s = 1
        for k in range(1, i + 1):
            s *= k
        # Step 13: Correct evaluation of DF[i] at x = 0
        a[i] = (-1)**(i+1) / s

    q[0] = 1
    p[0] = a[0]

    for i in range(1, N):
        for j in range(1, i):
            if j <= n:
                b[i][j] = 0
        if i <= n:
            b[i][i] = 1
        for j in range(i, N):
            b[i][j] = 0
        for j in range(1, i):
            if j <= m:
                b[i][n + j] = -a[i - j]
        for j in range(n + i, N):
            b[i][N-1] = a[i]

        # Find pivot and perform Gaussian elimination
        k = find_pivot(b, i)
        print(b)
        if b[k][i] == 0:
            print('The System is Singular first entry')
            return None, None
        if k != i:
            # Swap rows using NumPy slicing
            b[[i, k]] = b[[k, i]]

        # Elimination process
        for j in range(i + 1, N):
            xm = b[j][i] / b[i][i]
            for k in range(i, N + 1):
                b[j][k] -= xm * b[i][k]
            b[j][i] = 0

    if b[N - 1][N - 1] == 0:
        print('The System is Singular last entry')
        return None, None

    # Back substitution to compute q and p
    if m > 0:
        q[m - 1] = b[N - 1][N] / b[N - 1][N - 1]

    for i in range(N - 2, n - 1, -1):
        s = 0
        for k in range(i + 1, N):
            s += b[i][k] * q[k - n]
        q[i - n] = (b[i][N] - s) / b[i][i]

    for i in range(n - 1, -1, -1):
        s = 0
        for k in range(n, N):
            s += b[i][k] * p[k - n]
        p[i] = b[i][N] - s

    return p, q

# Define symbolic expressions list DF
DF = [1, sp.exp(-x), -sp.exp(-x), sp.exp(-x), -sp.exp(-x)]  # Symbolic expressions
m = 2
n = 3

# Call the PA function to execute the process
p, q = PA(f, n, m, DF)

# Output the results
if p is not None and q is not None:
    print("p:", p)
    print("q:", q)


In [ ]:
import numpy as np
from scipy.interpolate import pade
import sympy as sp

x=sp.symbols('x')

def r(p,q):
  for i in range(len(p)):
    rn=(p[i]*x**(i))
  for i in range(len(q)):
    rd=(q[i]*x**(i))
  p=rn/rd
  return p

DF = [1, sp.exp(-x), -sp.exp(-x), sp.exp(-x), -sp.exp(-x)]  # Symbolic expressions
m = 2
n = 3
N=m+n
a=[0]*(N+1)
for i in range(N+1):
  s = 1
  for k in range(1, i + 1):
      s *= k
  a[i] = (-1)**(i+1) / s
p, q = pade(a, m,n)
r=r(p,q)
r=sp.lambdify(x,r,'numpy')
print(r)
print('x     e^-x      P5(x)    |e-x-P5(x)|    r(x)         |e-x-r(x)|')

x=np.arange(0,1.2,0.2)
t2=np.exp(-x)
l=[]


print(r(0.2))

In [ ]:
import numpy as np
from scipy.interpolate import pade
import sympy as sp

# Define symbolic variable x
x = sp.symbols('x')

# Function to construct rational function r(p/q)
def r(p, q):
    # Generate numerator (rn) and denominator (rd) polynomials
    rn = sum(p[i] * x**i for i in range(len(p)))  # Sum for numerator
    rd = sum(q[i] * x**i for i in range(len(q)))  # Sum for denominator
    rational = rn / rd
    return rational

# Taylor coefficients for e^(-x) expansion
DF = [1, sp.exp(-x), -sp.exp(-x), sp.exp(-x), -sp.exp(-x)]  # Symbolic expressions
m = 2
n = 3
N = m + n
a = [0] * (N + 1)

# Construct the Taylor coefficients of e^(-x)
for i in range(N + 1):
    s = 1
    for k in range(1, i + 1):
        s *= k
    a[i] = (-1)**(i+1) / s

# Use scipy Pade approximation
p, q = pade(a, m, n)

# Create rational expression
r_expr = r(p, q)

# Lambdify the rational expression for numerical evaluation
r_func = sp.lambdify(x, r_expr, 'numpy')

# Display rational expression and check for correctness
print("Rational approximation r(x):", r_expr)

# Header for the table
print('x     e^-x      r(x)         |e-x-x|')

# Numerical evaluation points
x_values = np.arange(0.2, 1.2, 0.2)
t2 = np.exp(-x_values)

# List to hold values of r(x)
r_values = [r_func(val) for val in x_values]
# Printing out values
for x_val, exp_val, r_val in zip(x_values, t2, r_values):
    print(f"{x_val:.2f}  {exp_val:.5f}  {-1*r_val:.5f}  {abs(exp_val + r_val):.5f}")


In [ ]:
import numpy as np


m=3
n=2

N=m+n
A=np.zeros((N+1,N+1))
b=np.zeros((m,1))
a=np.array([1,-1,0.5,-1/6,1/24,-1/(24*5)])


for k in range(N+1):
  for i in range(k):
    A[k][i]=a[k-i]

print(A)